# Scheduler Simulation Dashboard

Interactive exploration notebook for `trace.json` files produced by the scheduler simulator.

## Setup

- Ensure the simulator has produced an up-to-date `trace.json` (default: `build/bin/Debug/results/trace.json`).
- This notebook assumes it is executed from the `python/` directory inside the repository.

In [1]:
pip install ipywidgets nbformat ipython plotly pandas

Note: you may need to restart the kernel to use updated packages.


In [52]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(NOTEBOOK_DIR))

import metrics_loader as ml

DEFAULT_TRACE = PROJECT_ROOT / "build" / "bin" / "Debug" / "results" / "trace.json"
trace_directory = DEFAULT_TRACE.parent

trace_files = sorted([path.name for path in trace_directory.glob("trace*.json")]) or [DEFAULT_TRACE.name]
trace_selector = widgets.Dropdown(options=trace_files, value=DEFAULT_TRACE.name, description="Trace:")
display(trace_selector)

trace = None
config = {}
summary = {}
tasks = None
ticks = None


Dropdown(description='Trace:', options=('trace.json', 'trace_fcfs.json', 'trace_mlfq.json', 'trace_rr.json', '…

In [53]:
selected_path = trace_directory / trace_selector.value
print(f'Using trace: {selected_path}')

trace = ml.load_trace(selected_path)
config = ml.config_dict(trace)
summary = ml.summary_dict(trace)

summary


Using trace: o:\Projects\Emebedded-Systems-CPU-Scheduler-Simulator\build\bin\Debug\results\trace_sjf.json


{'average_response_time_ms': 7.582946079516117,
 'average_runtime_ms': 2.9886422302529687,
 'average_turnaround_time_ms': 10.571955447370362,
 'average_wait_time_ms': 7.58182756840475,
 'completed_tasks': 13557,
 'core_idle_time_ms': [7923, 11555],
 'cpu_utilization': {'average': 0.6629166666666667, 'samples': 30000},
 'simulation_end_ms': 30001,
 'simulation_start_ms': 0,
 'task_count': 13566,
 'total_idle_time_ms': 19478,
 'total_simulation_time_ms': 30001}

In [54]:
config

{'base_power_watts': 6.5,
 'clock_speed_mhz': 1000.0,
 'context_switch_cost_us': 50,
 'idle_power_watts': 1.2,
 'io_completion_quantum_us': 300,
 'max_power_watts': 15.0,
 'num_cores': 2,
 'planned_run_duration_ms': 30000,
 'system_name': '2 Core IoT Compute Node',
 'tick_interval_us': 500,
 'verbose_logging': True}

In [55]:
import pandas as pd

tasks = ml.task_lifecycle_df(trace)
ticks = ml.ticks_df(trace)

tasks.head()

,arrival_ms,class,completion_ms,deadline_ms,dispatch_count,final_state,first_dispatch_ms,name,priority,requested_exec_max_ms,requested_exec_min_ms,runtime_ms,task_id,turnaround_time_ms,wait_time_ms
0,18690,rt,18692,10,1,completed,18690,wake_word_detection#1870,1,2,2,2,1869,2.0,0
1,12890,rt,12891,5,1,completed,12890,audio_processing#2579,2,1,1,1,5579,1.0,0
2,3625,rt,3627,5,1,completed,3625,audio_processing#726,2,2,2,2,3726,2.0,0
3,0,rt,2,10,1,completed,0,wake_word_detection#1,1,2,2,2,0,2.0,0
4,20757,interactive,20761,33,1,completed,20757,screen_refresh#630,7,4,4,4,13153,4.0,0


## Core schedule

In [56]:
core_fig = ml.make_core_timeline_figure(trace)
core_fig

## CPU utilisation over time

In [57]:
cpu_fig = ml.make_cpu_utilisation_figure(trace, rolling_window=200)
cpu_fig

## Core idle totals

In [58]:
idle_fig = ml.make_core_idle_bar_figure(trace)
idle_fig

## Task runtime vs wait

In [59]:
runtime_fig = ml.make_task_runtime_scatter(trace)
runtime_fig

## Additional exploration

In [60]:
tasks.groupby("class")["runtime_ms"].sum().sort_values(ascending=False)

class
rt             28586
interactive     7744
background      4193
Name: runtime_ms, dtype: int64